In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [0]:
rw_db_dataset_churn = spark.read.table("fiap_analytics.bronze.rw_db_dataset_churn")
display(rw_db_dataset_churn)

In [0]:
window_spec = Window.partitionBy("user_id").orderBy("timestamp")

rw_db_dataset_churn = rw_db_dataset_churn.withColumn(
    "next_session",
    lead("timestamp").over(window_spec)
)

rw_db_dataset_churn = rw_db_dataset_churn.withColumn(
    "days_until_next_session",
    datediff(col("next_session"), col("timestamp"))
)

rw_db_dataset_churn = rw_db_dataset_churn.withColumn(
    "churn",
    when(col("days_until_next_session") > 30, 1).otherwise(0)
)

display(rw_db_dataset_churn)

rw_db_dataset_churn.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fiap_analytics.silver.rw_db_dataset_churn")